In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# ============================================================================
# FRED Silver Layer
# ============================================================================
# Industry techniques demonstrated:
#   1. Streaming Silver from Auto Loader Bronze
#   2. Array explode (observations already parsed as array in Bronze)
#   3. Quarantine pattern for missing/invalid observations
#   4. DLT expectations as data quality contracts
#   5. Manifest-based audit trail (batch read from JSON files)
# ============================================================================

# ---- Paths for manifests (from your ingest notebook) ----
MANIFESTS_PATH = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/manifests/"


def _base():
    """
    TECHNIQUE: Streaming base reader with array explode
    With the explicit Bronze schema, observations is already a proper
    array of structs — no JSON string parsing needed. We simply
    explode the array to get one row per observation per series.
    """
    b = dlt.read_stream("canada_business.bronze.bronze_fred_raw_json")

    return (
        b.withColumn("obs", F.explode_outer(F.col("observations")))
         .withColumn("observation_date", F.to_date(F.col("obs.date")))
         .withColumn("value_raw", F.col("obs.value"))
         .withColumn(
             "value_double",
             F.when(F.col("value_raw").isin(".", "NA", ""), F.lit(None).cast("double"))
              .otherwise(F.col("value_raw").cast("double"))
         )
         .withColumn("obs_realtime_start", F.to_date(F.col("obs.realtime_start")))
         .withColumn("obs_realtime_end",   F.to_date(F.col("obs.realtime_end")))
         .withColumn("realtime_start",     F.to_date(F.col("realtime_start")))
         .withColumn("realtime_end",       F.to_date(F.col("realtime_end")))
    )


# ---------------- Silver: quarantine ----------------
@dlt.table(
  name="silver_fred_quarantine",
  comment="Rows that failed parsing or have missing/invalid fields"
)
def silver_fred_quarantine():
    x = _base()
    return (
        x.select(
            "series_id","_ingest_ts","_source_file",
            "observation_date","value_raw","value_double",
            "obs_realtime_start","obs_realtime_end","realtime_start","realtime_end",
            F.when(F.col("obs").isNull(), F.lit("missing_observation_object"))
             .when(F.col("observation_date").isNull(), F.lit("invalid_or_missing_date"))
             .otherwise(F.lit("other")).alias("reject_reason")
        )
        .where(
            F.col("obs").isNull() |
            F.col("observation_date").isNull()
        )
    )

# ---------------- Silver: clean observations ----------------
@dlt.table(
  name="silver_fred_observations",
  comment="Clean FRED observations (one row per series_id per date per ingest file)"
)
@dlt.expect_or_drop("valid_series", "series_id IS NOT NULL AND series_id <> ''")
@dlt.expect_or_drop("valid_date",   "observation_date IS NOT NULL")
def silver_fred_observations():
    x = _base()
    return x.select(
        "series_id","observation_date",
        "value_raw","value_double",
        "obs_realtime_start","obs_realtime_end",
        "realtime_start","realtime_end",
        "_ingest_ts","_source_file"
    )

# ---------------- Silver: basic metrics ----------------
@dlt.table(
  name="silver_fred_metrics",
  comment="Basic monitoring metrics"
)
def silver_fred_metrics():
    x = _base()
    return (
        x.groupBy()
         .agg(
             F.count("*").alias("rows_seen"),
             F.sum(F.when(F.col("obs").isNull(), 1).otherwise(0)).alias("missing_observation"),
             F.sum(F.when(F.col("observation_date").isNull(), 1).otherwise(0)).alias("invalid_date"),
             F.max(F.col("_ingest_ts")).alias("latest_ingest_ts")
         )
    )

# ---------------- Silver: ingest run audit (from manifests) ----------------
@dlt.table(
  name="silver_fred_runs",
  comment="One row per ingest run (from run_manifest_*.json)"
)
def silver_fred_runs():
    df = (
        spark.read
          .format("json")
          .option("multiLine", "true")
          .load(MANIFESTS_PATH)
    )

    per_series = df.select(
        "run_ts","catalog","schema","bronze_base","observation_start",
        F.explode_outer("series").alias("s")
    ).select(
        "run_ts","catalog","schema","bronze_base","observation_start",
        F.col("s.series_id").alias("series_id"),
        F.col("s.rows").cast("long").alias("rows"),
        F.col("s.raw_file").alias("raw_file")
    )

    return (
        per_series.groupBy("run_ts","catalog","schema","bronze_base","observation_start")
          .agg(
              F.countDistinct("series_id").alias("series_count"),
              F.countDistinct("raw_file").alias("file_count"),
              F.sum("rows").alias("total_rows")
          )
    )

In [0]:
# ---------------- Silver: Google Trends (pivot + score) ----------------
@dlt.table(
  name="silver_google_trends",
  comment="Pivoted Google Trends (one row per date) + demand_trend_score"
)
def silver_google_trends():
    b = dlt.read("canada_business.bronze.bronze_google_trends_raw")

    # Keep only latest ingest per (keyword, date)
    from pyspark.sql.window import Window
    w_dedup = Window.partitionBy("keyword", "date").orderBy(F.col("ingest_ts").desc())
    b = (
        b.withColumn("_rn", F.row_number().over(w_dedup))
         .where(F.col("_rn") == 1)
         .drop("_rn")
    )

    p = (
        b.groupBy("date")
         .agg(
             F.max(F.when(F.col("keyword") == "buy laptop", F.col("value"))).alias("laptop_idx"),
             F.max(F.when(F.col("keyword") == "TV deals", F.col("value"))).alias("tv_idx"),
             F.max(F.when(F.col("keyword") == "gaming console", F.col("value"))).alias("console_idx"),
             F.max(F.when(F.col("keyword") == "electronics sale", F.col("value"))).alias("electronics_sale_idx"),
         )
    )

    return (
        p.withColumn(
            "demand_trend_score",
            0.35*F.col("laptop_idx") +
            0.25*F.col("tv_idx") +
            0.25*F.col("console_idx") +
            0.15*F.col("electronics_sale_idx")
        )
        .withColumn("ingest_ts", F.current_timestamp())
    )

In [0]:

import dlt
from pyspark.sql import functions as F

# ============================================================================
# WITS Tariff Silver Layer
# ============================================================================
# Industry techniques demonstrated:
#   1. SDMX dimension-index decoding (standards-based metadata parsing)
#   2. Schema coercion: STRUCT → JSON → MAP for dynamic-key handling
#   3. Quarantine pattern (bad rows isolated, not dropped silently)
#   4. DLT expectations (declarative data quality contracts)
#   5. Outlier flagging with configurable thresholds
#   6. Natural-key deduplication for idempotent pipelines
#   7. Lineage tracking (source_file, ingest_ts on every row)
#   8. Pipeline observability metrics table
# ============================================================================

def _wits_base():
    """
    Core transformation: decodes SDMX JSON into a flat, analyst-friendly table.

    KEY TECHNIQUE — Schema coercion for semi-structured data:
    Spark's JSON schema inference treats JSON objects with dynamic keys as STRUCTs
    (one named field per key). But SDMX series keys like "0:0:0:12:0" are dynamic
    data, not a fixed schema. We convert the STRUCT back to a JSON string, then
    re-parse it with an explicit MAP schema. This is a standard workaround for
    Spark's greedy struct inference on dynamic-key JSON.
    """
    b = dlt.read_stream("canada_business.bronze.bronze_wits_tariff_raw")

    # ---- TECHNIQUE: Dimension lookup arrays from SDMX metadata ----
    # Each dimension (freq, reporter, partner, product, indicator) has an
    # ordered array of possible values in the structure block. We extract
    # the ID field from each to build a positional lookup.
    freq_vals      = F.expr("transform(structure.dimensions.series[0].values, x -> x.id)")
    reporter_vals  = F.expr("transform(structure.dimensions.series[1].values, x -> x.id)")
    partner_vals   = F.expr("transform(structure.dimensions.series[2].values, x -> x.id)")
    product_vals   = F.expr("transform(structure.dimensions.series[3].values, x -> x.id)")
    indicator_vals = F.expr("transform(structure.dimensions.series[4].values, x -> x.id)")

    # Time periods live in the observation dimension (not the series dimension)
    time_vals      = F.expr("transform(structure.dimensions.observation[0].values, x -> x.id)")

    # ---- TECHNIQUE: STRUCT → JSON string → MAP re-parse ----
    # Spark inferred the SDMX "series" object as a deeply nested STRUCT because
    # each series key (e.g. "0:0:0:12:0") became a named struct field.
    # We serialize it back to JSON, then re-parse with MAP<STRING, STRING> so we
    # can dynamically iterate over series keys using explode.
    series_json = F.to_json(F.col("dataSets").getItem(0).getField("series"))

    series_map = F.from_json(
        series_json,
        "MAP<STRING, STRING>"
    )

    # Explode the series map: one row per series key
    exploded = (
        b.withColumn("series_map", series_map)
         .select("*", F.explode(F.col("series_map")).alias("series_key", "series_value_json"))
    )

    # ---- TECHNIQUE: Nested observation parsing with explicit schema ----
    # Each series value contains {annotations, attributes, observations}.
    # observations is a MAP<STRING, ARRAY<DOUBLE>> where key = time index,
    # value = [tariff_rate, status_code]. We parse with explicit schema to
    # avoid further struct inference issues.
    obs_schema = "STRUCT<observations: MAP<STRING, ARRAY<DOUBLE>>>"

    parsed = (
        exploded
         .withColumn("series_parsed", F.from_json(F.col("series_value_json"), obs_schema))
         .withColumn("obs_map", F.col("series_parsed.observations"))
    )

    # ---- TECHNIQUE: Observation-level explosion ----
    # Explode observations: each entry is time_index → [value, status]
    # producing one row per (series × time period) combination.
    key_parts = F.split(F.col("series_key"), ":")

    x = (
        parsed
         .select("*", F.explode(F.map_entries(F.col("obs_map"))).alias("obs_entry"))
         .withColumn("time_idx", F.col("obs_entry.key").cast("int"))
         .withColumn("obs_arr",  F.col("obs_entry.value"))

         # Extract the tariff rate (first element of the observation array)
         .withColumn("tariff_value_raw", F.element_at(F.col("obs_arr"), 1))

         # ---- TECHNIQUE: Index-to-ID resolution ----
         # Map the time index back to year string using the lookup array.
         # element_at is 1-based, so we add 1 to the 0-based SDMX index.
         .withColumn("year_str", F.element_at(time_vals, F.col("time_idx") + 1))
         .withColumn("year", F.col("year_str").cast("int"))

         # Resolve each dimension index to its human-readable ID
         .withColumn("freq",        F.element_at(freq_vals,      F.element_at(key_parts, 1).cast("int") + 1))
         .withColumn("reporter",    F.element_at(reporter_vals,  F.element_at(key_parts, 2).cast("int") + 1))
         .withColumn("partner",     F.element_at(partner_vals,   F.element_at(key_parts, 3).cast("int") + 1))
         .withColumn("productcode", F.element_at(product_vals,   F.element_at(key_parts, 4).cast("int") + 1))
         .withColumn("indicator",   F.element_at(indicator_vals, F.element_at(key_parts, 5).cast("int") + 1))

         # ---- TECHNIQUE: Lineage tracking ----
         # Carry source_file and ingest_ts through every layer so any
         # row in Gold can be traced back to the exact raw file that produced it.
         .withColumn("source_file", F.col("source_file"))
         .withColumn("ingest_ts", F.current_timestamp())

         # Safe cast to double
         .withColumn("tariff_value", F.col("tariff_value_raw").cast("double"))

         # ---- TECHNIQUE: Outlier flagging ----
         # Rather than silently dropping suspicious values, we flag them
         # with boolean columns. This supports both quarantine routing
         # AND downstream analysis of data quality over time.
         .withColumn("is_negative", F.col("tariff_value").isNotNull() & (F.col("tariff_value") < 0))
         .withColumn("is_extreme",  F.col("tariff_value").isNotNull() & (F.col("tariff_value") > 200))
    )

    return x


# ============================================================================
# TECHNIQUE: Quarantine Pattern
# ----------------------------------------------------------------------------
# Industry best practice: never silently drop bad data. Instead, route it to
# a quarantine table with a machine-readable reject_reason. This lets data
# engineers audit failures, measure data quality trends, and replay/fix
# upstream issues without re-ingesting everything.
# ============================================================================
@dlt.table(
  name="silver_wits_tariff_quarantine",
  comment="Quarantined WITS tariff rows — missing values, bad year, outliers, parsing gaps"
)
def silver_wits_tariff_quarantine():
    x = _wits_base()

    return (
        x.withColumn(
            "reject_reason",
            # ---- TECHNIQUE: Multi-rule rejection classification ----
            # Each row gets exactly one reject_reason (first matching rule wins).
            # This makes quarantine analysis simple: GROUP BY reject_reason.
            F.when(F.col("year").isNull(), F.lit("missing_or_invalid_year"))
             .when(F.col("tariff_value").isNull(), F.lit("missing_or_invalid_tariff_value"))
             .when(F.col("reporter").isNull() | F.col("partner").isNull() |
                   F.col("productcode").isNull() | F.col("indicator").isNull(),
                   F.lit("missing_dimension_mapping"))
             .when(F.col("is_negative"), F.lit("negative_value_outlier"))
             .when(F.col("is_extreme"),  F.lit("extreme_value_outlier"))
             .otherwise(F.lit(None))
        )
        .where(F.col("reject_reason").isNotNull())
        .select(
            "freq","reporter","partner","productcode","indicator",
            "year","tariff_value_raw","tariff_value",
            "reject_reason","series_key","source_file","ingest_ts"
        )
    )


# ============================================================================
# TECHNIQUE: DLT Expectations (Declarative Data Quality Contracts)
# ----------------------------------------------------------------------------
# @dlt.expect_or_drop enforces schema-level contracts directly in the pipeline
# definition. Unlike ad-hoc WHERE clauses, expectations are:
#   - Visible in the DLT UI as pass/fail metrics per run
#   - Trackable over time for data quality SLAs
#   - Self-documenting (the contract IS the code)
# ============================================================================
@dlt.table(
  name="silver_wits_tariff",
  comment="Clean WITS tariff table — one row per reporter/partner/product/indicator/year"
)
@dlt.expect_or_drop("valid_year",  "year IS NOT NULL")
@dlt.expect_or_drop("valid_value", "tariff_value IS NOT NULL")
def silver_wits_tariff():
    x = _wits_base()

    clean = (
        x.where(
            (F.col("year").isNotNull()) &
            (F.col("tariff_value").isNotNull()) &
            (~F.col("is_negative")) &
            (~F.col("is_extreme"))
        )
        # ---- TECHNIQUE: Natural-key deduplication ----
        # If the same datapoint appears in multiple ingest files (e.g., two
        # raw JSON snapshots), deduplicate on the business key so downstream
        # joins and aggregations remain correct. This is critical for
        # idempotent pipelines where re-runs should not create duplicates.
        .dropDuplicates(["reporter","partner","productcode","indicator","year"])
    )

    return clean.select(
        "freq","reporter","partner","productcode","indicator",
        "year","tariff_value",
        "series_key","source_file","ingest_ts"
    )


# ============================================================================
# TECHNIQUE: Pipeline Observability Metrics
# ----------------------------------------------------------------------------
# A dedicated metrics table provides a single-row summary of each pipeline
# run. This feeds operational dashboards and alerting (e.g., "if missing_value
# count spikes >5% from last run, fire a PagerDuty alert").
# ============================================================================
@dlt.table(
  name="silver_wits_tariff_metrics",
  comment="Data quality metrics for WITS tariff dataset"
)
def silver_wits_tariff_metrics():
    # ---- TECHNIQUE: Batch read for aggregate metrics ----
    # Streaming aggregations without watermarks cannot use complete mode in DLT.
    # Metrics tables are inherently batch snapshots, so we read from the clean
    # silver table (already materialized) rather than re-streaming from bronze.
    clean = dlt.read("silver_wits_tariff")
    quarantine = dlt.read("silver_wits_tariff_quarantine")

    clean_metrics = (
        clean.groupBy()
         .agg(
            F.count("*").alias("clean_rows"),
            F.countDistinct("series_key").alias("series_keys"),
            F.max("ingest_ts").alias("latest_ingest_ts")
         )
    )

    quarantine_metrics = (
        quarantine.groupBy()
         .agg(
            F.count("*").alias("quarantined_rows"),
            F.countDistinct("reject_reason").alias("reject_reason_count")
         )
    )

    return clean_metrics.crossJoin(quarantine_metrics)

In [0]:

from pyspark.sql import functions as F

# ============================================================================
# StatCan Silver Layer
# ============================================================================
# Industry techniques demonstrated:
#   1. Multi-table routing from single bronze (PID-based filtering)
#   2. Government data cleaning (STATUS codes, SCALAR_FACTOR normalization)
#   3. Column name sanitization (verbose govt names → clean snake_case)
#   4. Quarantine pattern for StatCan symbol codes (F, x, .., etc.)
#   5. DLT expectations for data quality contracts
#   6. NAPCS/NAICS code filtering for domain-specific views
# ============================================================================

def _statcan_base():
    """
    TECHNIQUE: Unified base reader with government data normalization
    StatCan CSVs have verbose column names with spaces and parentheses.
    We normalize values and classify quality flags at the entry point
    so downstream Silver tables start from a consistent base.
    """
    b = dlt.read_stream("canada_business.bronze.bronze_statcan_raw")

    return (
        b
         # ---- TECHNIQUE: Scalar-adjusted value ----
         # StatCan VALUE is already decimal-adjusted, but SCALAR_FACTOR
         # tells us the multiplier (units, thousands, millions).
         # We normalize everything to actual units for consistency.
         # e.g., VALUE=41377 with SCALAR_ID=3 → 41,377,000 (thousands)
         .withColumn(
             "value_numeric",
             F.when(F.col("VALUE").isNull(), F.lit(None).cast("double"))
              .when(F.col("VALUE") == "", F.lit(None).cast("double"))
              .otherwise(F.col("VALUE").cast("double"))
         )
         .withColumn(
             "scalar_multiplier",
             F.when(F.col("SCALAR_ID") == "0", 1.0)          # units
              .when(F.col("SCALAR_ID") == "3", 1000.0)        # thousands
              .when(F.col("SCALAR_ID") == "6", 1000000.0)     # millions
              .when(F.col("SCALAR_ID") == "9", 1000000000.0)  # billions
              .otherwise(1.0)
         )
         .withColumn("value_adjusted", F.col("value_numeric") * F.col("scalar_multiplier"))
         # ---- TECHNIQUE: Status code classification ----
         # StatCan uses single-char symbol codes to flag data quality.
         # We map each to a human-readable quality_flag so downstream
         # tables can filter cleanly and quarantine can classify rejects.
         #   F = too unreliable to publish
         #   x = suppressed for confidentiality
         #   .. = not available for the reference period
         #   e = estimated value
         #   p = preliminary (subject to revision)
         #   r = revised since last release
         .withColumn(
             "quality_flag",
             F.when(F.col("STATUS") == "F", "unreliable")
              .when(F.col("STATUS") == "x", "suppressed")
              .when(F.col("STATUS").isin("..", "...", ""), "not_available")
              .when(F.col("STATUS") == "e", "estimated")
              .when(F.col("STATUS") == "p", "preliminary")
              .when(F.col("STATUS") == "r", "revised")
              .when(F.col("STATUS").isNull(), "ok")
              .otherwise("ok")
         )
         .withColumn("ingest_ts", F.current_timestamp())
    )


# ============================================================================
# SILVER 1: Quarantine — bad/suppressed StatCan rows
# ============================================================================
# TECHNIQUE: Quarantine pattern (same pattern as FRED and WITS Silver)
# Rows with unreliable, suppressed, or missing values are isolated here
# rather than silently dropped. This enables:
#   - Data quality monitoring (how many rows rejected per PID?)
#   - Audit trail (which STATUS codes caused rejection?)
#   - Reprocessing if thresholds change later
# ============================================================================
@dlt.table(
    name="silver_statcan_quarantine",
    comment="Quarantined StatCan rows — suppressed, unreliable, or unparseable values"
)
def silver_statcan_quarantine():
    x = _statcan_base()

    return (
        x.where(
            F.col("quality_flag").isin("unreliable", "suppressed", "not_available") |
            F.col("value_numeric").isNull()
        )
        .select(
            "pid", "REF_DATE", "GEO", "value_numeric", "STATUS",
            "quality_flag", "UOM", "SCALAR_FACTOR",
            "source_file", "ingest_ts"
        )
    )


# ============================================================================
# SILVER 2: Electronics Retail Sales (PID 20100056)
# ============================================================================
# TECHNIQUE: PID-based multi-table routing + NAICS code filtering
# All three StatCan tables land in one bronze table (618K rows).
# Silver splits them by PID and applies table-specific business logic.
#
# For retail sales we filter to:
#   - PID 20100056 (Monthly retail trade sales by NAICS)
#   - NAICS 4492 (Electronics and Appliances Retailers)
#   - GEO = Canada (national level, not provincial)
#   - quality_flag = ok (excludes suppressed/unreliable rows)
#
# TECHNIQUE: DLT expectations as declarative data quality contracts
# expect_or_drop enforces NOT NULL on retail_sales_cad and observation_date.
# Any rows that pass the WHERE filter but still have nulls are dropped
# and logged in the DLT event log for observability.
# ============================================================================
@dlt.table(
    name="silver_statcan_retail_electronics",
    comment="Monthly Canadian retail sales — Electronics & Appliances Retailers (NAICS 4492)"
)
@dlt.expect_or_drop("valid_value", "retail_sales_cad IS NOT NULL")
@dlt.expect_or_drop("valid_date",  "observation_date IS NOT NULL")
def silver_statcan_retail_electronics():
    x = _statcan_base()

    return (
        x.where(
            (F.col("pid") == "20100056") &
            # ---- TECHNIQUE: NAICS code matching with contains() ----
            # The NAICS column has verbose labels like:
            #   "Electronics and appliances retailers [4492]"
            # We use contains("4492") for flexible matching across
            # any label variations StatCan might introduce over time.
            (F.col("`North American Industry Classification System (NAICS)`").contains("4492")) &
            (F.col("GEO") == "Canada") &
            (F.col("quality_flag") == "ok")
        )
        .select(
            F.to_date(F.col("REF_DATE"), "yyyy-MM").alias("observation_date"),
            F.lit("CAN").alias("geo_code"),
            F.col("`North American Industry Classification System (NAICS)`").alias("naics_label"),
            F.col("Sales").alias("sales_type"),
            F.col("Adjustments").alias("adjustment_type"),
            F.col("value_adjusted").alias("retail_sales_cad"),
            F.col("UOM").alias("unit_of_measure"),
            F.col("SCALAR_FACTOR").alias("scalar_factor"),
            F.col("quality_flag"),
            F.col("pid"),
            F.col("source_file"),
            F.col("ingest_ts")
        )
    )


# ============================================================================
# SILVER 3: Inventory-to-Sales Ratio (PID 20100076)
# ============================================================================
# TECHNIQUE: Supply-side signal extraction
# The inventory-to-sales ratio is the single most important supply metric
# for the stock-up / reduce recommendation engine:
#   - High ratio (>1.5) → overstocked → reduce purchasing
#   - Low ratio (<1.0) → stockout risk → increase purchasing
#   - Normal range (1.0–1.5) → healthy inventory levels
#
# We keep all NAICS subsectors (not just electronics) so Gold can:
#   - Filter to electronics-specific ratio if available
#   - Fall back to total retail ratio if subsector data is sparse
#   - Compare electronics ratio vs overall retail for relative signals
# ============================================================================
@dlt.table(
    name="silver_statcan_inventory_ratio",
    comment="Monthly retail inventory-to-sales ratio — key supply health metric"
)
@dlt.expect_or_drop("valid_value", "inventory_sales_ratio IS NOT NULL")
@dlt.expect_or_drop("valid_date",  "observation_date IS NOT NULL")
def silver_statcan_inventory_ratio():
    x = _statcan_base()

    return (
        x.where(
            (F.col("pid") == "20100076") &
            (F.col("GEO") == "Canada") &
            (F.col("quality_flag") == "ok")
        )
        .select(
            F.to_date(F.col("REF_DATE"), "yyyy-MM").alias("observation_date"),
            F.lit("CAN").alias("geo_code"),
            F.col("`North American Industry Classification System (NAICS)`").alias("naics_label"),
            F.col("Adjustments").alias("adjustment_type"),
            F.col("value_adjusted").alias("inventory_sales_ratio"),
            F.col("UOM").alias("unit_of_measure"),
            F.col("quality_flag"),
            F.col("pid"),
            F.col("source_file"),
            F.col("ingest_ts")
        )
    )


# ============================================================================
# SILVER 4: Electronics Imports (PID 12100121)
# ============================================================================
# TECHNIQUE: NAPCS product code filtering for electronics imports
# The international merchandise trade table uses NAPCS classification
# (not HS codes like WITS). We filter using keyword matching on the
# NAPCS label to capture all electronics-related categories:
#   - Computers, Electronic, Electrical, Semiconductor
#   - Telecommunication, Office machinery, Machinery
#   - NAPCS codes [34x] and [35x] (machinery & electrical equipment)
#
# TECHNIQUE: Trade flow filtering
# The table contains both imports AND exports. We filter to imports
# only since we're measuring the supply pipeline flowing INTO Canada.
# This gives us a leading indicator of future inventory availability —
# if import volumes drop, inventory shortages may follow in 1-3 months.
#
# TECHNIQUE: Consistent product domain across pipeline
# The NAPCS electronics categories here align with:
#   - WITS tariff HS 84-85 (Machinery & Electronics)
#   - StatCan retail NAICS 4492 (Electronics & Appliances Retailers)
# This ensures the supply-demand-tariff signals are measuring the
# same product category end-to-end.
# ============================================================================
@dlt.table(
    name="silver_statcan_imports_electronics",
    comment="Monthly Canadian imports — Machinery & Electronics (NAPCS)"
)
@dlt.expect_or_drop("valid_value", "import_value_cad IS NOT NULL")
@dlt.expect_or_drop("valid_date",  "observation_date IS NOT NULL")
def silver_statcan_imports_electronics():
    x = _statcan_base()

    napcs_col = F.col("`North American Product Classification System (NAPCS)`")

    return (
        x.where(
            (F.col("pid") == "12100121") &
            (F.col("Trade").contains("Import")) &
            (
                napcs_col.contains("Computers") |
                napcs_col.contains("Electronic") |
                napcs_col.contains("Electrical") |
                napcs_col.contains("Semiconductor") |
                napcs_col.contains("Telecommunication") |
                napcs_col.contains("Office machinery") |
                napcs_col.contains("Machinery") |
                napcs_col.contains("[34") |
                napcs_col.contains("[35")
            ) &
            (F.col("GEO") == "Canada") &
            (F.col("quality_flag") == "ok")
        )
        .select(
            F.to_date(F.col("REF_DATE"), "yyyy-MM").alias("observation_date"),
            F.lit("CAN").alias("geo_code"),
            F.col("Trade").alias("trade_flow"),
            napcs_col.alias("napcs_label"),
            F.col("`Seasonal adjustment`").alias("seasonal_adjustment"),
            F.col("Basis").alias("basis"),
            F.col("value_adjusted").alias("import_value_cad"),
            F.col("UOM").alias("unit_of_measure"),
            F.col("SCALAR_FACTOR").alias("scalar_factor"),
            F.col("quality_flag"),
            F.col("pid"),
            F.col("source_file"),
            F.col("ingest_ts")
        )
    )


# ============================================================================
# SILVER 5: StatCan Metrics (observability)
# ============================================================================
# TECHNIQUE: Per-PID quality metrics for pipeline observability
# Breaks down data quality by source table (PID) so you can spot
# issues in one dataset without them being masked by the others.
# Uses approx_count_distinct for streaming compatibility — exact
# countDistinct is not supported on streaming DataFrames in DLT.
#
# This table answers questions like:
#   - How many rows per table were ingested?
#   - What % of trade data is suppressed for confidentiality?
#   - Are we getting new reference periods each run?
# ============================================================================
@dlt.table(
    name="silver_statcan_metrics",
    comment="Data quality metrics per StatCan table"
)
def silver_statcan_metrics():
    x = _statcan_base()

    return (
        x.groupBy("pid")
         .agg(
            F.count("*").alias("rows_seen"),
            F.sum(F.when(F.col("value_numeric").isNull(), 1).otherwise(0)).alias("null_values"),
            F.sum(F.when(F.col("quality_flag") == "unreliable", 1).otherwise(0)).alias("unreliable"),
            F.sum(F.when(F.col("quality_flag") == "suppressed", 1).otherwise(0)).alias("suppressed"),
            F.sum(F.when(F.col("quality_flag") == "not_available", 1).otherwise(0)).alias("not_available"),
            F.approx_count_distinct("REF_DATE").alias("approx_periods"),
            F.max("ingest_ts").alias("latest_ingest_ts")
         )
    )